In [ ]:
import findspark
findspark.init()

In [ ]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, DateType

spark = SparkSession.builder. \
    appName("pyspark-assesment"). \
    getOrCreate()

### Read data

In [10]:
df = spark.read.csv("/dataset/nyc-jobs.csv", header=True)
df.printSchema()

root
 |-- Job ID: string (nullable = true)
 |-- Agency: string (nullable = true)
 |-- Posting Type: string (nullable = true)
 |-- # Of Positions: string (nullable = true)
 |-- Business Title: string (nullable = true)
 |-- Civil Service Title: string (nullable = true)
 |-- Title Code No: string (nullable = true)
 |-- Level: string (nullable = true)
 |-- Job Category: string (nullable = true)
 |-- Full-Time/Part-Time indicator: string (nullable = true)
 |-- Salary Range From: string (nullable = true)
 |-- Salary Range To: string (nullable = true)
 |-- Salary Frequency: string (nullable = true)
 |-- Work Location: string (nullable = true)
 |-- Division/Work Unit: string (nullable = true)
 |-- Job Description: string (nullable = true)
 |-- Minimum Qual Requirements: string (nullable = true)
 |-- Preferred Skills: string (nullable = true)
 |-- Additional Information: string (nullable = true)
 |-- To Apply: string (nullable = true)
 |-- Hours/Shift: string (nullable = true)
 |-- Work Locatio

### Sample function

In [48]:
def get_salary_frequency(df: DataFrame) -> list:
    row_list = df.select('Salary Frequency').distinct().collect()
    return [row['Salary Frequency'] for row in row_list]

### Example of test function

In [65]:
mock_data = [('A', 'Annual'), ('B', 'Daily')]
expected_result = ['Annual', 'Daily']

In [66]:
def test_get_salary_frequency(mock_data: list, 
                              expected_result: list,
                              schema: list = ['id', 'Salary Frequency']):  
    mock_df = spark.createDataFrame(data = mock_data, schema = schema)
    assert get_salary_frequency(mock_df) == expected_result

---
# Data Exploration – Detailed Analysis

Below we analyze column types, categorical columns, nulls, and basic statistics.

In [ ]:
# 1. Schema and column types
df.printSchema()
print("\nTotal rows:", df.count())
print("Total columns:", len(df.columns))

In [ ]:
# 2. Column classification: Numerical vs Character (all read as string from CSV)
# Columns that are inherently numerical (stored as string in raw CSV):
numeric_cols_raw = ["# Of Positions", "Salary Range From", "Salary Range To", "Title Code No"]
# Categorical columns (low cardinality, used for grouping):
categorical_cols = ["Agency", "Posting Type", "Level", "Job Category", "Full-Time/Part-Time indicator", "Salary Frequency"]
# Date columns:
date_cols = ["Posting Date", "Post Until", "Posting Updated", "Process Date"]
# Text/long character columns (for profiling, not for direct aggregation):
text_cols = ["Job Description", "Minimum Qual Requirements", "Preferred Skills", "Additional Information", "Business Title", "Civil Service Title"]

print("Numeric (raw):", numeric_cols_raw)
print("Categorical:", categorical_cols)
print("Date:", date_cols)
print("Text:", text_cols)

In [ ]:
# 3. Null counts per column
from pyspark.sql.functions import col, count, when
null_counts = df.select([count(when(col(c).isNull() | (col(c) == ""), 1)).alias(c) for c in df.columns])
null_counts.show(vertical=True)

In [ ]:
# 4. Distinct counts for categorical columns (cardinality)
for c in categorical_cols:
    if c in df.columns:
        cnt = df.select(c).distinct().count()
        print(f"{c}: {cnt} distinct values")

---
# KPIs

We build an enriched dataframe with typed salary and date for KPI computations. Salary is normalized to annual for comparison (Hourly/Daily converted to approximate annual).

In [ ]:
# Helper: cast numeric and parse posting date for reuse across KPIs
def prepare_for_kpis(df: DataFrame) -> DataFrame:
    """Cast salary columns to double, add posting year and annual salary for analysis."""
    from pyspark.sql.functions import coalesce, when, to_date
    df = df.withColumn("Salary Range From", F.col("Salary Range From").cast(DoubleType())) \
           .withColumn("Salary Range To", F.col("Salary Range To").cast(DoubleType())) \
           .withColumn("Posting Date", to_date(F.col("Posting Date"), "yyyy-MM-dd"))
    # Approximate annual salary: use midpoint of range; for non-Annual, scale (Hourly*2080, Daily*260)
    df = df.withColumn("_mid_salary", (F.col("Salary Range From") + F.col("Salary Range To")) / 2)
    df = df.withColumn(
        "annual_salary",
        when(F.lower(F.col("Salary Frequency")).contains("annual"), F.col("_mid_salary"))
        .when(F.lower(F.col("Salary Frequency")).contains("hour"), F.col("_mid_salary") * 2080)
        .when(F.lower(F.col("Salary Frequency")).contains("day"), F.col("_mid_salary") * 260)
        .otherwise(F.col("_mid_salary"))
    ).drop("_mid_salary")
    df = df.withColumn("posting_year", F.year(F.col("Posting Date")))
    return df

df_kpi = prepare_for_kpis(df)
df_kpi.select("Salary Range From", "Salary Range To", "Salary Frequency", "annual_salary", "posting_year", "Posting Date").show(5, truncate=False)

### KPI 1: Number of job postings per category (Top 10)

In [ ]:
def get_top10_jobs_per_category(df: DataFrame) -> DataFrame:
    """Returns Top 10 job categories by number of postings."""
    return df.groupBy("Job Category").count().orderBy(F.desc("count")).limit(10)

kpi1 = get_top10_jobs_per_category(df_kpi)
kpi1.show(truncate=False)

### KPI 2: Salary distribution per job category

In [ ]:
def get_salary_distribution_per_category(df: DataFrame) -> DataFrame:
    """Avg/min/max annual salary per job category (excluding null salaries)."""
    return df.filter(F.col("annual_salary").isNotNull() & (F.col("annual_salary") > 0)) \
        .groupBy("Job Category") \
        .agg(
            F.count("*").alias("count"),
            F.avg("annual_salary").alias("avg_salary"),
            F.min("annual_salary").alias("min_salary"),
            F.max("annual_salary").alias("max_salary")
        ).orderBy(F.desc("avg_salary"))

kpi2 = get_salary_distribution_per_category(df_kpi)
kpi2.show(20, truncate=False)

### KPI 3: Correlation between higher degree and salary

We derive a simple "degree level" from Minimum Qual Requirements: baccalaureate/master/phd = higher, high school = lower, then compare average salary.

In [ ]:
def add_degree_level(df: DataFrame) -> DataFrame:
    """Add column degree_level: 2=graduate/baccalaureate+, 1=high school or equivalent, 0=unknown."""
    from pyspark.sql.functions import lower, when
    mq = lower(F.col("Minimum Qual Requirements"))
    return df.withColumn(
        "degree_level",
        when(mq.contains("baccalaureate") | mq.contains("bachelor") | mq.contains("graduate") | mq.contains("master") | mq.contains("phd") | mq.contains("degree"), 2)
        .when(mq.contains("high school") | mq.contains("equivalent"), 1)
        .otherwise(0)
    )

def get_salary_by_degree_level(df: DataFrame) -> DataFrame:
    """Average annual salary by degree level (for correlation view)."""
    df_deg = add_degree_level(df).filter(F.col("annual_salary").isNotNull() & (F.col("annual_salary") > 0))
    return df_deg.groupBy("degree_level").agg(
        F.count("*").alias("count"),
        F.avg("annual_salary").alias("avg_salary")
    ).orderBy("degree_level")

df_with_degree = add_degree_level(df_kpi)
kpi3 = get_salary_by_degree_level(df_kpi)
kpi3.show()
# Correlation: compare avg salary across degree levels (higher degree_level => higher education requirement)
print("\nConclusion: If avg_salary increases with degree_level, there is positive correlation between higher degree and salary.")

### KPI 4: Job posting with the highest salary per agency

In [ ]:
from pyspark.sql.window import Window
def get_highest_salary_posting_per_agency(df: DataFrame) -> DataFrame:
    """One row per agency: the job posting with the highest annual salary."""
    w = Window.partitionBy("Agency").orderBy(F.desc("annual_salary"))
    return df.filter(F.col("annual_salary").isNotNull() & (F.col("annual_salary") > 0)) \
        .withColumn("rank", F.row_number().over(w)) \
        .filter(F.col("rank") == 1) \
        .select("Agency", "Business Title", "Job Category", "annual_salary", "Salary Frequency") \
        .drop("rank")

kpi4 = get_highest_salary_posting_per_agency(df_kpi)
kpi4.show(20, truncate=25)

### KPI 5: Job postings average salary per agency for the last 2 years

In [ ]:
def get_avg_salary_per_agency_last_n_years(df: DataFrame, n_years: int = 2) -> DataFrame:
    """Average annual salary per agency for postings in the last n years (from max posting year)."""
    max_year = df.agg(F.max("posting_year")).collect()[0][0]
    min_year = max_year - n_years + 1
    return df.filter((F.col("posting_year") >= min_year) & F.col("annual_salary").isNotNull() & (F.col("annual_salary") > 0)) \
        .groupBy("Agency") \
        .agg(F.avg("annual_salary").alias("avg_salary_last_2y"), F.count("*").alias("posting_count")) \
        .orderBy(F.desc("avg_salary_last_2y"))

kpi5 = get_avg_salary_per_agency_last_n_years(df_kpi, 2)
kpi5.show(20, truncate=False)

### KPI 6: Highest paid skills (in this dataset – NYC jobs)

We parse Preferred Skills (and optionally Minimum Qual Requirements) into skill-like tokens and compute average salary per skill. *Note: Dataset is NYC jobs; we interpret "US market" as this NYC job market.*

In [ ]:
# Explode Preferred Skills into skill phrases (split by comma/semicolon/common delimiters)
from pyspark.sql.functions import explode, split, trim, lower
def get_highest_paid_skills(df: DataFrame, top_n: int = 15) -> DataFrame:
    """Tokenize Preferred Skills, compute avg annual salary per skill, return top N by avg salary."""
    skills_df = df.filter(F.col("Preferred Skills").isNotNull() & (F.col("Preferred Skills") != "") 
                          & F.col("annual_salary").isNotNull() & (F.col("annual_salary") > 0)) \
        .withColumn("_skills", split(F.regexp_replace(F.col("Preferred Skills"), "[;•·]", ","), ",")) \
        .withColumn("skill", explode("_skills")) \
        .withColumn("skill", trim(lower(F.col("skill")))) \
        .filter((F.length(F.col("skill")) >= 3) & (F.length(F.col("skill")) <= 80))
    return skills_df.groupBy("skill") \
        .agg(F.avg("annual_salary").alias("avg_salary"), F.count("*").alias("count")) \
        .filter(F.col("count") >= 2) \
        .orderBy(F.desc("avg_salary")) \
        .limit(top_n)

kpi6 = get_highest_paid_skills(df_kpi, 15)
kpi6.show(15, truncate=False)

---
# Visualizations

In [ ]:
# Use matplotlib for compatibility in Docker/Jupyter
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [ ]:
# Viz 1: Top 10 job categories by posting count
k1 = kpi1.toPandas()
fig, ax = plt.subplots(figsize=(10, 5))
cats = k1["Job Category"].fillna("(blank)").tolist()
counts = k1["count"].tolist()
ax.barh(range(len(cats)), counts, color="steelblue", alpha=0.8)
ax.set_yticks(range(len(cats)))
ax.set_yticklabels(cats, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Number of postings")
ax.set_title("KPI 1: Top 10 job categories by posting count")
plt.tight_layout()
plt.show()

In [ ]:
# Viz 2: Salary distribution per job category (top 10 by avg salary)
k2 = kpi2.limit(10).toPandas()
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(k2))
ax.bar([i - 0.2 for i in x], k2["min_salary"], width=0.2, label="Min", color="lightblue")
ax.bar(x, k2["avg_salary"], width=0.2, label="Avg", color="steelblue")
ax.bar([i + 0.2 for i in x], k2["max_salary"], width=0.2, label="Max", color="darkblue", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(k2["Job Category"].fillna("(blank)"), rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Annual salary ($)")
ax.set_title("KPI 2: Salary distribution per job category (top 10)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Viz 3: Average salary by degree level (0=unknown, 1=HS, 2=bachelor+)
k3 = kpi3.toPandas()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(k3["degree_level"].astype(str), k3["avg_salary"], color=["gray", "orange", "green"], alpha=0.8)
ax.set_xlabel("Degree level (0=unknown, 1=HS, 2=bachelor/graduate+)")
ax.set_ylabel("Average annual salary ($)")
ax.set_title("KPI 3: Correlation – average salary by degree level")
plt.tight_layout()
plt.show()

In [ ]:
# Viz 4: Highest paid skills (top 10)
k6 = kpi6.limit(10).toPandas()
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(k6)), k6["avg_salary"], color="teal", alpha=0.8)
ax.set_yticks(range(len(k6)))
ax.set_yticklabels(k6["skill"].str[:45], fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("Average annual salary ($)")
ax.set_title("KPI 6: Highest paid skills (NYC job postings)")
plt.tight_layout()
plt.show()

---
# Data Processing

Functions for cleaning, column pre-processing, data wrangling, transformation, feature engineering, and feature removal. Processed data is written to a target file.

In [ ]:
def clean_string_column(df: DataFrame, col_name: str) -> DataFrame:
    """Trim whitespace and replace empty strings with null for a string column."""
    return df.withColumn(col_name, F.when(F.trim(F.col(col_name)) == "", None).otherwise(F.trim(F.col(col_name))))

def clean_numeric_column(df: DataFrame, col_name: str) -> DataFrame:
    """Cast to double; replace negative or invalid with null (zero kept for e.g. # Of Positions)."""
    from pyspark.sql.types import DoubleType
    return df.withColumn(col_name, F.col(col_name).cast(DoubleType())) \
             .withColumn(col_name, F.when((F.col(col_name).isNull()) | (F.col(col_name) < 0), None).otherwise(F.col(col_name)))

In [ ]:
def preprocess_dates(df: DataFrame) -> DataFrame:
    """Parse date columns and add posting_year, posting_month for analytics."""
    from pyspark.sql.functions import to_date, year, month
    for c in ["Posting Date", "Post Until", "Posting Updated", "Process Date"]:
        if c in df.columns:
            df = df.withColumn(c, to_date(F.col(c), "yyyy-MM-dd"))
    df = df.withColumn("posting_year", year(F.col("Posting Date"))) \
           .withColumn("posting_month", month(F.col("Posting Date")))
    return df

In [ ]:
# Feature engineering 1: Annual salary (normalized for comparison across Salary Frequency)
def add_annual_salary(df: DataFrame) -> DataFrame:
    """Add annual_salary: midpoint of range, scaled to annual for Hourly/Daily."""
    df = df.withColumn("Salary Range From", F.col("Salary Range From").cast(DoubleType())) \
           .withColumn("Salary Range To", F.col("Salary Range To").cast(DoubleType()))
    mid = (F.col("Salary Range From") + F.col("Salary Range To")) / 2
    df = df.withColumn("annual_salary",
        F.when(F.lower(F.col("Salary Frequency")).contains("annual"), mid)
         .when(F.lower(F.col("Salary Frequency")).contains("hour"), mid * 2080)
         .when(F.lower(F.col("Salary Frequency")).contains("day"), mid * 260)
         .otherwise(mid))
    return df

# Feature engineering 2: Degree level from Minimum Qual Requirements (0/1/2)
# Note: earlier KPI section defines `add_degree_level`; keep the same name here for consistency.
def add_degree_level(df: DataFrame) -> DataFrame:
    """Add degree_level: 2=graduate/baccalaureate+, 1=high school, 0=unknown."""
    mq = F.lower(F.col("Minimum Qual Requirements"))
    return df.withColumn("degree_level",
        F.when(mq.contains("baccalaureate") | mq.contains("bachelor") | mq.contains("graduate") | mq.contains("master") | mq.contains("phd") | mq.contains("degree"), 2)
         .when(mq.contains("high school") | mq.contains("equivalent"), 1)
         .otherwise(0))

# Feature engineering 3: Salary band (low / medium / high) for segmentation
def add_salary_band(df: DataFrame) -> DataFrame:
    """Add salary_band based on annual_salary percentiles (computed from non-null)."""
    # Use simple thresholds: <50k low, 50k-100k medium, >100k high (can be replaced by quantiles)
    return df.withColumn("salary_band",
        F.when(F.col("annual_salary").isNull(), "unknown")
         .when(F.col("annual_salary") < 50000, "low")
         .when(F.col("annual_salary") <= 100000, "medium")
         .otherwise("high"))

In [ ]:
# Features to remove based on exploration: very high null rate or redundant
# - Recruitment Contact, Work Location 1 (often duplicate/empty)
# - To Apply, Hours/Shift (low analytical value for salary/category KPIs)
# - Post Until, Posting Updated (we keep Posting Date and Process Date)
COLUMNS_TO_DROP = ["Recruitment Contact", "Work Location 1", "To Apply", "Hours/Shift", "Post Until", "Posting Updated"]

def drop_low_value_columns(df: DataFrame, columns_to_drop: list = None) -> DataFrame:
    """Remove columns that add little value for modeling/analytics (from profiling)."""
    to_drop = columns_to_drop or COLUMNS_TO_DROP
    for c in to_drop:
        if c in df.columns:
            df = df.drop(c)
    return df

In [ ]:
def process_dataset(df: DataFrame, target_path: str = "/dataset/nyc-jobs-processed") -> DataFrame:
    """
    Full pipeline: clean, preprocess dates, add annual_salary, degree_level, salary_band,
    drop selected columns, and write processed data to target_path (parquet).
    Returns the processed DataFrame.
    """
    # Cleaning
    for c in ["Agency", "Job Category", "Business Title", "Salary Frequency"]:
        if c in df.columns:
            df = clean_string_column(df, c)
    for c in ["# Of Positions", "Salary Range From", "Salary Range To", "Title Code No"]:
        if c in df.columns:
            df = clean_numeric_column(df, c)
    # Preprocessing
    df = preprocess_dates(df)
    df = add_annual_salary(df)
    df = add_degree_level(df)
    df = add_salary_band(df)
    df = drop_low_value_columns(df)
    # Write to target (overwrite)
    df.write.mode("overwrite").parquet(target_path)
    return df

In [ ]:
# Run full processing and save to target file
TARGET_PATH = "/dataset/nyc-jobs-processed"
df_processed = process_dataset(df, TARGET_PATH)
print("Processed schema:")
df_processed.printSchema()
print("\nRow count:", df_processed.count())
df_processed.show(3, truncate=20)

---
# Test Cases

In [ ]:
# Run existing sample test
test_get_salary_frequency(mock_data, expected_result)
print("test_get_salary_frequency: PASSED")

In [ ]:
def test_get_top10_jobs_per_category():
    small = spark.createDataFrame([("A", "IT"), ("B", "IT"), ("C", "HR")], ["id", "Job Category"])
    result = get_top10_jobs_per_category(small)
    rows = result.collect()
    assert len(rows) <= 10
    it_row = [r for r in rows if r["Job Category"] == "IT"][0]
    assert it_row["count"] == 2
    print("test_get_top10_jobs_per_category: PASSED")
test_get_top10_jobs_per_category()

In [ ]:
def test_add_degree_level():
    data = [
        ("1", "baccalaureate degree required", 50000.0),
        ("2", "high school or equivalent", 40000.0),
        ("3", "other", 45000.0)
    ]
    small = spark.createDataFrame(data, ["id", "Minimum Qual Requirements", "annual_salary"])
    out = add_degree_level(small)
    rows = out.select("degree_level").collect()
    levels = [r["degree_level"] for r in rows]
    assert 2 in levels and 1 in levels and 0 in levels
    print("test_add_degree_level: PASSED")
test_add_degree_level()

In [ ]:
def test_process_dataset_drops_columns():
    """Processed DataFrame should not contain removed columns."""
    sample = df.limit(10)
    out = process_dataset(sample, "/dataset/nyc-jobs-processed-test")
    for c in COLUMNS_TO_DROP:
        assert c not in out.columns, f"Column {c} should have been dropped"
    print("test_process_dataset_drops_columns: PASSED")
test_process_dataset_drops_columns()

In [ ]:
def test_processed_has_new_features():
    """Processed data should have annual_salary, degree_level, salary_band, posting_year."""
    sample = df.limit(5)
    out = process_dataset(sample, "/dataset/nyc-jobs-processed-test2")
    required = ["annual_salary", "degree_level", "salary_band", "posting_year"]
    for f in required:
        assert f in out.columns, f"Missing feature: {f}"
    print("test_processed_has_new_features: PASSED")
test_processed_has_new_features()

---
# Attendance report: days in office per month

Goal: for each `EMPID`, compute how many **distinct days** they were present in the office in a given month (e.g., April 2026).

In [ ]:
from pyspark.sql import functions as F

# Example input: one row per employee with a comma-separated list of dates present
attendance = [
    (1, "2026-04-01,2026-04-02,2026-04-03"),
    (2, "2026-04-02,2026-04-04,2026-04-05,2026-04-06"),
    (3, "2026-04-01,2026-04-02,2026-04-04,2026-04-05,2026-04-07,2026-04-09"),
]

df_att = spark.createDataFrame(attendance, ["EMPID", "DatePresent"])


def office_days_per_month(
    df,
    emp_col: str = "EMPID",
    dates_col: str = "DatePresent",
    year: int = 2026,
    month: int = 4,
):
    """Return per-employee count of distinct office days in a given year/month.

    Assumes `dates_col` is a comma-separated list of ISO dates (yyyy-MM-dd).
    If your raw data is already one row per day, skip split/explode and just filter + countDistinct.
    """

    exploded = (
        df.withColumn("_date", F.explode(F.split(F.col(dates_col), ",")))
        .withColumn("_date", F.to_date(F.trim(F.col("_date")), "yyyy-MM-dd"))
        .filter(F.col("_date").isNotNull())
        .filter((F.year(F.col("_date")) == F.lit(year)) & (F.month(F.col("_date")) == F.lit(month)))
    )

    return (
        exploded.groupBy(emp_col)
        .agg(F.countDistinct("_date").alias("TotalDays"))
        .orderBy(emp_col)
    )


report_apr_2026 = office_days_per_month(df_att, year=2026, month=4)
report_apr_2026.show(truncate=False)

In [ ]:
# Variant: if your source data is already one row per employee per day
# schema example: (EMPID, DatePresent) where DatePresent is a proper date column or yyyy-MM-dd string

def office_days_per_month_row_per_day(
    df,
    emp_col: str = "EMPID",
    date_col: str = "DatePresent",
    year: int = 2026,
    month: int = 4,
):
    df2 = df.withColumn("_date", F.to_date(F.col(date_col), "yyyy-MM-dd"))
    return (
        df2.filter(F.col("_date").isNotNull())
        .filter((F.year(F.col("_date")) == F.lit(year)) & (F.month(F.col("_date")) == F.lit(month)))
        .groupBy(emp_col)
        .agg(F.countDistinct("_date").alias("TotalDays"))
        .orderBy(emp_col)
    )

---
# Deployment & Trigger (summary)

- **Deployment**: Use Docker Compose as in INSTALL.md (`docker compose -f ./docker-compose.yml --project-name my_assesment up`). Processed output is written to `/dataset/nyc-jobs-processed` (Parquet).
- **Trigger**: Run all cells in this notebook (Run All), or execute via `jupyter nbconvert --execute /notebook/assesment_notebook.ipynb` inside the Jupyter container. For scheduled runs, see **MyDocument.md** (Airflow, GitHub Actions, or cron).